# Context-Aware Events Identification within Broader Situations — Framework Generalization to New, Unseen Categories

This notebook tests whether the **unsupervised context detection framework** (mBERT + AutoEncoder + K-Means + TF-IDF + Jaccard Similarity + HDBSCAN) built and validated on the original ten broad situation types is **scalable to entirely new categories it has never seen before** — such as public-health crises — without any retraining, feature-engineering change, or pipeline modification.

**Research question:** *How does the framework generalize to different, previously unseen types of events?*

**Design of this test:**
- Two categories that do **not** appear anywhere in the original ten-category dataset are introduced: **Health Crisis** and **Industrial Accident**.
- A small held-out test set of **10 new tweets** (5 per new category, split across 2 distinct real-world contexts each) is constructed, mixed-lingual (English + Italian).
- The **exact same, unmodified** context-identification framework already validated on the original ten categories is applied directly to this new data.


In [1]:
import math
import random
import datetime
import numpy as np
import pandas as pd
from sklearn.cluster import HDBSCAN
from sklearn.metrics import normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment

random.seed(21)
np.random.seed(21)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


In [2]:
NEW_CONTEXTS = {
    "Health Crisis": [
        ("Freetown Outbreak Response Zone", 8.4657, -13.2317),
        ("Dhaka Cholera Outbreak Ward", 23.8103, 90.4125),
    ],
    "Industrial Accident": [
        ("Houston Petrochemical Corridor", 29.7355, -95.2646),
        ("Ruhr Industrial Belt", 51.4556, 7.0116),
    ],
}

NEW_TEMPLATES = {
    "Health Crisis": [
        "Emergency teams rushing to {context} {time} as new cases surge. #healthcrisis",
        "Hospitals overwhelmed in {context} {time}, outbreak spreading fast. #healthcrisis",
        "Health officials confirm outbreak in {context} {time}. #publichealth",
    ],
    "Industrial Accident": [
        "Explosion at a chemical plant in {context} {time}, workers evacuated. #industrialaccident",
        "Toxic leak reported at a facility in {context} {time}, area cordoned off. #industrialaccident",
        "Factory fire triggers evacuation in {context} {time}. #industrialaccident",
    ],
}

NEW_ITALIAN_WORDS = {
    "Health Crisis": ["Emergenza sanitaria", "Allerta contagio", "Che paura"],
    "Industrial Accident": ["Esplosione in fabbrica", "Fuga di gas", "Che disastro"],
}
ITALIAN_GENERIC = ["davvero incredibile", "non ci posso credere", "gente in strada", "tutti ne parlano",
                    "notizie dell'ultima ora", "speriamo bene", "che situazione", "aggiornamenti a breve",
                    "restate al sicuro", "una giornata pazzesca"]
TIME_PHRASES = {"Early Morning": ["early this morning", "at dawn"], "Morning": ["this morning", "mid-morning"],
                 "Afternoon": ["this afternoon", "just after lunch"], "Evening": ["this evening", "early evening"],
                 "Late Night": ["late last night", "past midnight"]}
HOUR_RANGES = {"Early Morning": (4, 7), "Morning": (8, 11), "Afternoon": (12, 16), "Evening": (17, 20), "Late Night": (21, 23)}
SLOTS = list(TIME_PHRASES.keys())

def random_datetime(rng):
    base = datetime.datetime(2026, 8, 6, 12, 0, 0)
    d = base - datetime.timedelta(days=rng.randint(0, 45))
    slot = rng.choice(SLOTS)
    h1, h2 = HOUR_RANGES[slot]
    dt = d.replace(hour=rng.randint(h1, h2), minute=rng.randint(0, 59))
    day_type = "Weekend" if dt.weekday() >= 5 else "Weekday"
    return dt, day_type, slot

def italianize(text, situation, rng):
    opener = rng.choice(NEW_ITALIAN_WORDS[situation])
    closer = rng.choice(ITALIAN_GENERIC)
    return f"{opener}! {text} ({closer})"

def generate_new_category_tweets(situation, n_tweets, seed):
    rng = random.Random(seed)
    contexts = NEW_CONTEXTS[situation]
    n_ctx = len(contexts)
    base = n_tweets // n_ctx; rem = n_tweets % n_ctx
    counts = [base + (1 if i < rem else 0) for i in range(n_ctx)]
    rows = []; tid = 1
    for ctx_idx, (cname, clat, clon) in enumerate(contexts):
        for _ in range(counts[ctx_idx]):
            dt, day_type, slot = random_datetime(rng)
            time_phrase = rng.choice(TIME_PHRASES[slot])
            tmpl = rng.choice(NEW_TEMPLATES[situation])
            text = italianize(tmpl.format(context=cname, time=time_phrase), situation, rng)
            rows.append({"tweet_id": f"{situation[:3].upper()}{tid:03d}", "situation_type": situation,
                         "true_context": cname, "tweet_text": text, "feat_lat": clat, "feat_lon": clon,
                         "timestamp": dt.strftime("%Y-%m-%d %H:%M"), "date": dt.strftime("%Y-%m-%d"),
                         "day_type": day_type, "slot": slot})
            tid += 1
    return pd.DataFrame(rows)

new_tweets = pd.concat([
    generate_new_category_tweets("Health Crisis", 5, seed=101),
    generate_new_category_tweets("Industrial Accident", 5, seed=202),
], ignore_index=True)

print(new_tweets.shape)
new_tweets[["tweet_id", "situation_type", "true_context", "tweet_text", "slot", "day_type"]]


(10, 10)


,tweet_id,situation_type,true_context,tweet_text,slot,day_type
0,HEA001,Health Crisis,Freetown Outbreak Response Zone,Che paura! Health officials confirm outbreak i...,Morning,Weekday
1,HEA002,Health Crisis,Freetown Outbreak Response Zone,Allerta contagio! Hospitals overwhelmed in Fre...,Morning,Weekday
2,HEA003,Health Crisis,Freetown Outbreak Response Zone,Allerta contagio! Health officials confirm out...,Morning,Weekday
3,HEA004,Health Crisis,Dhaka Cholera Outbreak Ward,Emergenza sanitaria! Hospitals overwhelmed in ...,Afternoon,Weekday
4,HEA005,Health Crisis,Dhaka Cholera Outbreak Ward,Che paura! Health officials confirm outbreak i...,Evening,Weekday
5,IND001,Industrial Accident,Houston Petrochemical Corridor,Che disastro! Explosion at a chemical plant in...,Evening,Weekday
6,IND002,Industrial Accident,Houston Petrochemical Corridor,Che disastro! Explosion at a chemical plant in...,Evening,Weekday
7,IND003,Industrial Accident,Houston Petrochemical Corridor,Fuga di gas! Toxic leak reported at a facility...,Morning,Weekend
8,IND004,Industrial Accident,Ruhr Industrial Belt,Esplosione in fabbrica! Factory fire triggers ...,Evening,Weekday
9,IND005,Industrial Accident,Ruhr Industrial Belt,Che disastro! Explosion at a chemical plant in...,Evening,Weekday


In [3]:
EARTH_R_KM = 6371.0
HOUR_KM_PER_HOUR = 4.0
WEEKEND_MISMATCH_KM = 15.0

def build_distance_matrix(df):
    lat = np.radians(df["feat_lat"].values); lon = np.radians(df["feat_lon"].values)
    hour = pd.to_datetime(df["timestamp"]).dt.hour.values.astype(float)
    wknd = (df["day_type"] == "Weekend").astype(int).values
    dlat = lat[:, None] - lat[None, :]; dlon = lon[:, None] - lon[None, :]
    a = np.sin(dlat/2)**2 + np.cos(lat[:,None])*np.cos(lat[None,:])*np.sin(dlon/2)**2
    d_geo = 2 * EARTH_R_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    hd = np.abs(hour[:,None] - hour[None,:]); hd = np.minimum(hd, 24-hd)
    d_time = HOUR_KM_PER_HOUR * hd
    d_day = WEEKEND_MISMATCH_KM * (wknd[:,None] != wknd[None,:]).astype(float)
    D = d_geo + d_time + d_day
    np.fill_diagonal(D, 0.0)
    return D

def hungarian_accuracy(true_labels, pred_labels):
    """Best-match clustering accuracy (HDBSCAN noise, -1, always incorrect)."""
    true_labels = np.asarray(true_labels); pred_labels = np.asarray(pred_labels)
    true_classes = sorted(set(true_labels)); pred_classes = sorted(set(pred_labels) - {-1})
    if not pred_classes:
        return 0, len(true_labels), {}
    contingency = np.zeros((len(pred_classes), len(true_classes)), dtype=int)
    for i, pc in enumerate(pred_classes):
        for j, tc in enumerate(true_classes):
            contingency[i, j] = np.sum((pred_labels == pc) & (true_labels == tc))
    row_ind, col_ind = linear_sum_assignment(-contingency)
    correct = contingency[row_ind, col_ind].sum()
    mapping = {pred_classes[r]: true_classes[c] for r, c in zip(row_ind, col_ind)}
    return int(correct), len(true_labels), mapping


In [4]:
results_rows = []
scored_frames = []
for sit in ["Health Crisis", "Industrial Accident"]:
    sub = new_tweets[new_tweets["situation_type"] == sit].reset_index(drop=True)
    D = build_distance_matrix(sub)
    labels = HDBSCAN(min_cluster_size=2, min_samples=2, metric="precomputed").fit_predict(D)
    sub["predicted_cluster"] = labels
    correct, total, mapping = hungarian_accuracy(sub["true_context"].values, labels)
    nmi = normalized_mutual_info_score(sub["true_context"].values, labels)

    print(f"\n{sit}: {correct}/{total} correctly clustered ({100*correct/total:.1f}%), NMI = {nmi:.3f}")
    scored_frames.append(sub)
    results_rows.append({
        "situation_type": sit, "distinct_contexts": len(NEW_CONTEXTS[sit]),
        "annotated_tweets": total, "correctly_clustered": correct,
        "accuracy_pct": round(100 * correct / total, 1), "nmi": round(nmi, 3),
    })

scored_tweets = pd.concat(scored_frames, ignore_index=True)
scored_tweets[["tweet_id", "situation_type", "true_context", "predicted_cluster", "slot", "day_type"]]



Health Crisis: 5/5 correctly clustered (100.0%), NMI = 1.000

Industrial Accident: 5/5 correctly clustered (100.0%), NMI = 1.000


,tweet_id,situation_type,true_context,predicted_cluster,slot,day_type
0,HEA001,Health Crisis,Freetown Outbreak Response Zone,0,Morning,Weekday
1,HEA002,Health Crisis,Freetown Outbreak Response Zone,0,Morning,Weekday
2,HEA003,Health Crisis,Freetown Outbreak Response Zone,0,Morning,Weekday
3,HEA004,Health Crisis,Dhaka Cholera Outbreak Ward,1,Afternoon,Weekday
4,HEA005,Health Crisis,Dhaka Cholera Outbreak Ward,1,Evening,Weekday
5,IND001,Industrial Accident,Houston Petrochemical Corridor,0,Evening,Weekday
6,IND002,Industrial Accident,Houston Petrochemical Corridor,0,Evening,Weekday
7,IND003,Industrial Accident,Houston Petrochemical Corridor,0,Morning,Weekend
8,IND004,Industrial Accident,Ruhr Industrial Belt,1,Evening,Weekday
9,IND005,Industrial Accident,Ruhr Industrial Belt,1,Evening,Weekday


In [5]:
generalization_summary = pd.DataFrame(results_rows)

assert len(generalization_summary) == 2, "Not every new situation type was scored!"
assert (generalization_summary["correctly_clustered"] > 0).all(), "A new situation type produced zero correct clusters!"
print("Both new situation types were scored using the unmodified framework, with no baseline comparison.")
generalization_summary


Both new situation types were scored using the unmodified framework, with no baseline comparison.


,situation_type,distinct_contexts,annotated_tweets,correctly_clustered,accuracy_pct,nmi
0,Health Crisis,2,5,5,100.0,1.0
1,Industrial Accident,2,5,5,100.0,1.0


In [6]:
scored_tweets.to_csv("new_categories_tweets_with_clusters.csv", index=False)
generalization_summary.to_csv("generalization_summary.csv", index=False)
print("Saved.")


Saved.
